In [ ]:
import os
import tensorflow as tf
import numpy as np
os.environ['CUDA_VISIBLE_DEVICES'] = '0,5'

#AdaptiveFocalLoss
# 针对类别不平衡二分类问题设计的损失函数
class AdaptiveFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma, alpha, delta, reduction=tf.keras.losses.Reduction.AUTO, name='AdaptiveFocalLoss'):
        super().__init__(reduction=reduction, name=name)
        self.gamma = gamma
        self.alpha = alpha
        self.delta = delta
    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        p_t = tf.clip_by_value(p_t, self.delta, 1.0 - self.delta)
        alpha_factor = tf.where(tf.equal(y_true, 1), self.alpha, 1 - self.alpha)
        focal_loss = -alpha_factor * tf.pow(1 - p_t, self.gamma) * tf.math.log(p_t)
        return tf.reduce_mean(focal_loss)
    def get_config(self):
        config = super().get_config()
        config.update({"gamma": self.gamma, "alpha": self.alpha, "delta": self.delta})
        return config

class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', threshold=0.5, **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        tp = tf.reduce_sum(y_true * y_pred)
        fp = tf.reduce_sum(y_pred) - tp
        fn = tf.reduce_sum(y_true) - tp
        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)
    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + 1e-6)
        recall = self.true_positives / (self.true_positives + self.false_negatives + 1e-6)
        return 2 * (precision * recall) / (precision + recall + 1e-6)
    def reset_states(self):
        self.true_positives.assign(0)
        self.false_positives.assign(0)
        self.false_negatives.assign(0)




In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.regularizers import l2





# ============================
# 多尺度 ConvNeXtBlock1D
# ============================
class ConvNeXtBlock1D(layers.Layer):
    def __init__(self, filters, kernel_sizes=[3, 5, 7], drop_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_sizes = kernel_sizes
        self.drop_rate = drop_rate
        self.conv_dw_branches = []
        for ks in kernel_sizes:
            self.conv_dw_branches.append(
                layers.Conv1D(filters, ks, padding='same', kernel_regularizer=l2(1e-4))
            )
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.mlp_dense1 = layers.Dense(filters * 4, activation='gelu', kernel_regularizer=l2(1e-4))
        self.mlp_dense2 = layers.Dense(filters, kernel_regularizer=l2(1e-4))
        self.dropout = layers.Dropout(drop_rate)
        self.proj = None

    def build(self, input_shape):
        in_channels = input_shape[-1]
        if in_channels != self.filters:
            self.proj = layers.Conv1D(self.filters, kernel_size=1, padding='same', kernel_regularizer=l2(1e-4))
        super().build(input_shape)

    def call(self, x, training=None):
        residual = x
        multi_out = self.conv_dw_branches[0](x)
        for branch in self.conv_dw_branches[1:]:
            multi_out = multi_out + branch(x)
        x = multi_out
        x = self.norm(x)
        x = self.mlp_dense1(x)
        x = self.mlp_dense2(x)
        x = self.dropout(x, training=training)
        if self.proj is not None:
            residual = self.proj(residual)
        return tf.nn.relu(x + residual)

# ======================== 模型构建 ========================
def build_optimized_model(input_shape, lr):
    inputs = layers.Input(shape=input_shape)
    x = ConvNeXtBlock1D(256, kernel_sizes=[3, 5, 7], drop_rate=0.1)(inputs)
    x = layers.MaxPooling1D(2)(x)
    x = layers.BatchNormalization()(x)
    x = ConvNeXtBlock1D(128, kernel_sizes=[3, 5, 7], drop_rate=0.1)(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Bidirectional(
        layers.GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                   kernel_regularizer=l2(1e-4), recurrent_regularizer=l2(1e-4))
    )(x)
    x = layers.Bidirectional(
        layers.GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2,
                   kernel_regularizer=l2(1e-4), recurrent_regularizer=l2(1e-4))
    )(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, kernel_regularizer=l2(1e-4))(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, kernel_regularizer=l2(1e-4))(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=AdaptiveFocalLoss(gamma=2.0, alpha=0.5, delta=0.02),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.Precision(name='precision'),
                 F1Score()]
    )
    return model

In [3]:
import numpy as np
import os

def save_results(indices, chr,res):
    # os.makedirs('no_results_25', exist_ok=True)     ##  ✅
    # np.savetxt(f'no_results_25/{chr}_boundary.txt', indices, fmt='%d')    # ✅
        # 动态创建文件夹：no_results_25 / no_results_50 / no_results_100
    folder_name = f'no_results_{res}'
    os.makedirs(folder_name, exist_ok=True)
    save_path = f'{folder_name}/{chr}_boundary.txt'
    np.savetxt(save_path, indices, fmt='%d')

def direct_predict_with_contiguous_processing(model, P_test, middle_row_indices_test, chr,res, threshold=0.5):

    """
    带连续段处理的直接预测函数，包含精确率、召回率和F1分数计算。

    参数：
    model -- 训练好的机器学习模型
    P_test -- 测试特征矩阵 (n_samples, n_features)
    middle_row_indices_test -- 基因组位置索引数组
    y_test -- 测试集标签 (n_samples,)
    chr -- 染色体标识符
    threshold -- 概率阈值（默认0.5）
    """
    # 1. 模型预测与概率映射
    y_pred_proba = model.predict(P_test).flatten()
    prob_map = {idx: prob for idx, prob in zip(middle_row_indices_test, y_pred_proba)}
    
    # 2. 筛选并排序候选索引
    selected = middle_row_indices_test[y_pred_proba > threshold]
    sorted_indices = np.sort(np.unique(selected))

    # 3. 连续段检测与处理
    final_indices = []
    current_segment = []

    for idx in sorted_indices:
        if not current_segment:
            current_segment.append(idx)
        else:
            if idx == current_segment[-1] + 1:
                current_segment.append(idx)
            else:
                if len(current_segment) >= 1:
                    max_prob_idx = max(current_segment, key=lambda x: prob_map[x])
                    final_indices.append(max_prob_idx)
                current_segment = [idx]

    if current_segment:
        max_prob_idx = max(current_segment, key=lambda x: prob_map[x])
        final_indices.append(max_prob_idx)


    #  保存结果
    save_results(np.array(final_indices), chr,res)
    return None

# 计算节点的度数
def compute_node_degrees(H):
    return np.sum(H, axis=1)

# 计算超边的度数
def compute_hyperedge_degrees(H):
    return np.sum(H, axis=0)

# 计算一阶转移概率
def compute_first_order_transition_probabilities(H, node_degrees, hyperedge_degrees):
    num_nodes, num_hyperedges = H.shape
    P1 = np.zeros((num_nodes, num_nodes))

    # 计算每个超边的贡献，只考虑非零元素
    for v in range(num_nodes):
        for u in range(num_nodes):
            if u == v:
                continue

            # 计算转移概率
            pi_uv = 0
            for e in range(num_hyperedges):
                if H[u, e] == 0 or H[v, e] == 0:  # 若没有连接，跳过
                    continue

                h_ve = H[v, e]  # 超边 e 与节点 v 的连接强度
                h_ue = H[u, e]  # 超边 e 与节点 u 的连接强度
                d_v = node_degrees[v]  # 节点 v 的度数
                delta_e = hyperedge_degrees[e]  # 超边 e 的度数

                # 一阶转移概率公式
                pi_uv += (h_ve * h_ue) / (d_v * delta_e)

            P1[v, u] = pi_uv

    return P1

# 计算二阶转移的α值
def compute_alpha(x, v, u, H, p, q):
    if x == u:
        return 1 / p
    elif np.any(H[x, :] * H[u, :]):  # x和u有共同超边
        return 1
    else:
        return 1 / q

# 计算二阶转移概率
def compute_second_order_transition_probabilities(H, P1, p, q):
    num_nodes = H.shape[0]
    P2 = np.zeros((num_nodes, num_nodes, num_nodes))

    for v in range(num_nodes):
        for u in range(num_nodes):
            for x in range(num_nodes):
                alpha_val = compute_alpha(x, v, u, H, p, q)
                P2[v, u, x] = alpha_val * P1[v, x]

    return P2


In [4]:
import numpy as np

def process_files_to_arrays(filenames):
    X_all_chr = []
    middle_row_indices_all_chr = []
    # y_all_chr = []
    sample_counts = []  # 用于记录每条染色体样本数

    for filename in filenames:
        count = 0
        with open(filename, 'r') as file:
            lines = file.readlines()
            for line in lines:
                line = line.strip()
                if line:
                    data = eval(line, {"array": np.array})
                    X_all_chr.append(data[0])
                    middle_row_indices_all_chr.append(data[1])
                    # y_all_chr.append(data[2])
                    count += 1
        sample_counts.append(count)

    return (np.array(X_all_chr),
            np.array(middle_row_indices_all_chr),
            # np.array(y_all_chr),
            sample_counts)

In [ ]:
# 输入 shape（训练时写死的）
input_shape = (11, 11)

lr=0.0001
# 构建模型
model = build_optimized_model(input_shape,lr)

_ = model(tf.zeros((1, *input_shape)))

weights_path = '...path.../checkpoints/seed_4203/lr_0.0001/best_model_seed_4203_lr_0.0001.weights.h5'

model.load_weights(weights_path)
print("权重加载成功！")



In [ ]:
import os
import numpy as np

# 参数定义
dir1 = '...path.../GM12878/model_data'   # ✅
#dir2 = '100_sub_matrix.txt'      # ✅
# 定义染色体和计数
res_file_list =  [
    '25_sub_matrix.txt',
    '50_sub_matrix.txt',
    '100_sub_matrix.txt'
]
chr_list = ['chr20','chr21','chr22']


In [ ]:
for chr in chr_list:
    if chr == 'chr3' or chr == 'chr18' :
        continue
    for dir2 in res_file_list:
        list1 = []
        f1 = os.path.join(dir1, f"{chr}_{dir2}")
        list1.append(f1)
        #print(chr)
        print(f"===== 正在处理：{chr} | 分辨率文件：{dir2} =====")
        print(f1)
        print(list1)
        X_test, middle_row_indices_test,sample_counts_test = process_files_to_arrays(list1)
        P_test= []
        for H in X_test:
        # p = 2  # 随便选择的参数值
        # q = 3  # 随便选择的参数值
            num_nodes, num_hyperedges = H.shape
        # 计算节点和超边的度数
            node_degrees = compute_node_degrees(H)
            hyperedge_degrees = compute_hyperedge_degrees(H)
        # 计算一阶转移概率
            P1 = compute_first_order_transition_probabilities(H, node_degrees, hyperedge_degrees)
            P_test.append(P1)
        P_test = np.array(P_test)  # 将列表转换为 NumPy 数组
        print("P_test shape:", P_test.shape)
    # direct_predict_simple(model,P_test,middle_row_indices_test,chr,threshold=0.5)

        res_num = dir2.split('_')[0]  #新增
        #direct_predict_with_contiguous_processing(model,P_test,middle_row_indices_test,chr,res_num,threshold=0.5)
        direct_predict_with_contiguous_processing(model, P_test, middle_row_indices_test, chr,res_num, threshold=0.5)
